# Radiant RAG MCP — Audio Ingestion & Summarization Test Notebook

End-to-end test of the `ingest_audio` MCP tool and `VideoSummarizationAgent`
applied to audio-only files.  Synthetic test audio files are generated in-process
using `espeak` TTS so the suite runs without any external media.

| | |
|---|---|
| **Transport** | Streamable HTTP |
| **Storage** | ChromaDB (embedded) |
| **Transcription** | Whisper (`base` model) |
| **Summarization** | `VideoSummarizationAgent` (brief / standard / detailed) |
| **LLM** | Ollama Cloud — `gemma4:31b-cloud` |

**Sections that require an LLM key are marked `[LLM]`.**  
Add `OLLAMA_API_KEY` to Colab Secrets (key icon in the left sidebar) to enable them.

---

## Run order

```
1a → (auto restart) → 1b → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11
→ [LLM] 12 → [LLM] 13 → [LLM] 14 → [LLM] 15 → [LLM] 16 → 17 → Summary
```


## 1  Install

Installs `radiant-rag-mcp` with the ChromaDB backend, audio dependencies
(faster-whisper, ffmpeg-python), embedding/reranking models, and helpers.

The cell ends with a **kernel restart** so freshly installed `.so` files are
loaded correctly.  After the restart, continue from **cell 1b**.


In [ ]:
import subprocess, sys, os

CONFIG_PATH = '/content/config.yaml'

# ── Step 1: FFmpeg (standard apt version is sufficient for audio) ──────────
!apt-get install -y -q ffmpeg espeak
!ffmpeg -version 2>&1 | head -1

# ── Step 2: Detect PyTorch CUDA tag ──────────────────────────────────────
import torch as _torch
_torch_cuda = _torch.version.cuda or ''
_cu_tag = f"cu{_torch_cuda.replace('.','')[:3]}" if _torch_cuda else ''
print(f'torch {_torch.__version__}  CUDA {_torch_cuda}  ->  tag: {_cu_tag}')
del _torch

# ── Step 3: Core RAG package ──────────────────────────────────────────────
!pip install -q --upgrade --no-cache-dir "radiant-rag-mcp[chroma] @ git+https://github.com/dshipley71/radiant-rag-mcp.git"
!pip install -q --prefer-binary nest_asyncio httpx "fastmcp>=3.0"
!wget -q "https://raw.githubusercontent.com/dshipley71/radiant-rag-mcp/main/config.yaml" -O {CONFIG_PATH}

# ── Step 4: Audio extras ──────────────────────────────────────────────────
!pip install -q ffmpeg-python "faster-whisper>=1.0.0"

# ── Step 5: Repair numpy/scipy/sklearn ───────────────────────────────────
!pip install -q --force-reinstall numpy scipy scikit-learn
!pip install -q --upgrade "transformers>=4.45.0" "sentence-transformers>=2.7"

# ── Step 6: Pillow ────────────────────────────────────────────────────────
!pip install -q --force-reinstall "Pillow>=11.0"

# ── Step 7: Restart ──────────────────────────────────────────────────────
print('All installs complete. Restarting kernel ...')
import time; time.sleep(1)
os.kill(os.getpid(), 9)


### ↻ Kernel restarted — continue from cell 1b

All package installs completed. Run **cell 1b** to verify imports and pre-cache models.


In [ ]:
# ── Cell 1b: runs after the automatic kernel restart ─────────────────────
import sys, warnings
from pathlib import Path

CONFIG_PATH = '/content/config.yaml'

# 1 ── Verify numpy ────────────────────────────────────────────────────────
import numpy as np
print(f'numpy  {np.__version__}')
try:
    import numpy._core.strings
    print('numpy  OK')
except (ImportError, AttributeError) as _e:
    raise RuntimeError(f'numpy inconsistent ({_e}). Re-run 1a → restart → re-run 1b.')

# 2 ── Verify scipy ────────────────────────────────────────────────────────
import scipy
print(f'scipy  {scipy.__version__}  OK')

# 3 ── Verify Pillow ───────────────────────────────────────────────────────
import PIL
print(f'Pillow  {PIL.__version__}  OK')

# 4 ── Pre-cache embedding / reranking models ─────────────────────────────
from sentence_transformers import SentenceTransformer, CrossEncoder
SentenceTransformer('sentence-transformers/all-MiniLM-L12-v2')
print('sentence-transformers  OK')
CrossEncoder('cross-encoder/ms-marco-MiniLM-L12-v2')
print('cross-encoder  OK')

print('\nInstall complete')


## 2  Import checks

Verifies every dependency the audio pipeline needs is importable.
A `MISSING` line means the package failed to install in section 1.


In [ ]:
import sys
import importlib
from pathlib import Path

# ── Required third-party packages ─────────────────────────────────────────
REQUIRED = [
    ('fastmcp',               'fastmcp'),
    ('nest_asyncio',          'nest_asyncio'),
    ('chromadb',              'chromadb'),
    ('sentence_transformers', 'sentence-transformers'),
    ('faster_whisper',        'faster-whisper'),
    ('ffmpeg',                'ffmpeg-python'),
]

# ── Radiant audio submodules ──────────────────────────────────────────────
AUDIO_CORE = [
    ('radiant_rag_mcp',                            'radiant-rag-mcp'),
    ('radiant_rag_mcp.ingestion.video_processor',  'see diagnostic below'),
    ('radiant_rag_mcp.agents.video_summarization', 'see diagnostic below'),
]

_missing_required = []
_missing_audio    = {}

print('=== Required packages ===')
for module, pkg in REQUIRED:
    try:
        __import__(module)
        print(f'  ✓       {module}')
    except ImportError:
        print(f'  ✗  {module}  ->  pip install {pkg}')
        _missing_required.append(pkg)

print()
print('=== Radiant RAG audio submodules ===')
for module, _ in AUDIO_CORE:
    try:
        __import__(module)
        print(f'  ✓       {module}')
    except Exception as e:
        print(f'  ✗     {module}')
        _missing_audio[module] = e

if _missing_audio:
    print()
    for mod, exc in _missing_audio.items():
        print(f'  Module : {mod}')
        print(f'  Error  : {type(exc).__name__}: {exc}')
        cause = exc.__cause__ or exc.__context__
        if cause:
            print(f'  Caused by: {type(cause).__name__}: {cause}')
        print()

if _missing_required:
    raise ImportError(f'Missing required packages: {", ".join(_missing_required)}')

if _missing_audio:
    import warnings
    warnings.warn(f'Audio submodules not importable: {list(_missing_audio)}. Sections 8-16 will fail.',
                  stacklevel=1)
else:
    print()
    print('All imports ok')


## 3  Configuration

Sets every environment variable the server needs before startup.
Audio-specific keys use the prefix `RADIANT_VIDEO_WHISPER_*`.

> Add `OLLAMA_API_KEY` to **Colab Secrets** (key icon, left sidebar) before running.


In [ ]:
import os

# Read LLM key from Colab Secrets, falling back to env
try:
    from google.colab import userdata
    _key = userdata.get('OLLAMA_API_KEY') or ''
except Exception:
    _key = os.environ.get('RADIANT_OLLAMA_API_KEY', '')

os.environ.update({
    'RADIANT_OLLAMA_BASE_URL':               'https://ollama.com/v1',
    'RADIANT_OLLAMA_API_KEY':                _key,
    'RADIANT_TRANSPORT':                     'http',
    'RADIANT_HOST':                          '127.0.0.1',
    'RADIANT_PORT':                          '8765',
    'RADIANT_STORAGE_BACKEND':               'chroma',
    'RADIANT_CONFIG_PATH':                   CONFIG_PATH,
    # Pipeline
    'RADIANT_PIPELINE_USE_CRITIC':           'false',
    'RADIANT_CITATION_ENABLED':              'false',
    'RADIANT_LLM_BACKEND_TIMEOUT':           '120',
    'RADIANT_LLM_BACKEND_MAX_RETRIES':       '0',
    'RADIANT_RERANKING_BACKEND_DEVICE':      'cuda',
    'RADIANT_EMBEDDING_BACKEND_DEVICE':      'cuda',
    'RADIANT_CHUNKING_USE_LLM_CHUNKING':     'false',
    'RADIANT_RETRIEVAL_DENSE_TOP_K':         '5',
    'RADIANT_RETRIEVAL_BM25_TOP_K':          '5',
    # Whisper transcription settings
    'RADIANT_VIDEO_WHISPER_MODEL':           'base',
    'RADIANT_VIDEO_WHISPER_DEVICE':          'auto',
    'RADIANT_VIDEO_WHISPER_COMPUTE_TYPE':    'int8',
    'RADIANT_VIDEO_WHISPER_LANGUAGE':        'auto',
    # Summarization defaults (overridden per-cell in sections 13-16)
    'RADIANT_VIDEO_SUMMARIZATION_SUMMARY_DETAIL':           'standard',
    'RADIANT_VIDEO_SUMMARIZATION_WINDOW_CAPTION_SENTENCES': '4',
})

SERVER_URL  = f"http://{os.environ['RADIANT_HOST']}:{os.environ['RADIANT_PORT']}/mcp"
HAS_LLM_KEY = bool(_key)

print('=== Configuration ===')
print(f'  Server URL      : {SERVER_URL}')
print(f'  LLM key set     : {HAS_LLM_KEY}')
print(f'  Whisper model   : {os.environ["RADIANT_VIDEO_WHISPER_MODEL"]}')
print(f'  Whisper device  : {os.environ["RADIANT_VIDEO_WHISPER_DEVICE"]}')
print(f'  Summary detail  : {os.environ["RADIANT_VIDEO_SUMMARIZATION_SUMMARY_DETAIL"]}')

if not HAS_LLM_KEY:
    print()
    print('[NOTE] OLLAMA_API_KEY not found. [LLM] cells will be skipped.')
    print('       Add it to Colab Secrets (key icon, left sidebar).')


## 4  Shared helpers

Defines:
- `run(tool, args)` — call an MCP tool and pretty-print the JSON result
- `assert_ok(result)` — raise `AssertionError` on tool errors
- `skip_llm()` — guard used in every `[LLM]` cell
- `_wait_for_server()` — async HTTP probe used by section 5


In [ ]:
import asyncio
import json
import logging
import threading
import time
import textwrap
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()  # Allow asyncio.run() inside Jupyter/Colab event loop

import httpx as _httpx
from fastmcp import Client
from fastmcp.exceptions import ToolError


async def _call(tool: str, args: dict = None):
    """Async helper: open a short-lived MCP client, call the tool, return parsed result."""
    try:
        async with Client(SERVER_URL) as client:
            raw = await client.call_tool(tool, args or {})
    except ToolError as e:
        return {'status': 'error', 'tool': tool, 'message': str(e)}
    except Exception as e:
        return {'status': 'error', 'tool': tool, 'message': f'{type(e).__name__}: {e}'}

    content = raw.content if hasattr(raw, 'content') else (raw or [])
    if not content:
        return None
    item = content[0]
    text = item.text if hasattr(item, 'text') else str(item)
    try:
        return json.loads(text)
    except (json.JSONDecodeError, ValueError):
        return text


def run(tool: str, args: dict = None):
    """Synchronous wrapper around _call; pretty-prints and returns the result."""
    result = asyncio.run(_call(tool, args))
    print(json.dumps(result, indent=2, default=str))
    return result


def skip_llm() -> bool:
    """Return True (and print a notice) when no LLM key is available."""
    if not HAS_LLM_KEY:
        print('[SKIPPED] Set OLLAMA_API_KEY in Colab Secrets to run this cell.')
        return True
    return False


def assert_ok(result, field: str = None):
    """Assert the tool result is not an error dict; optionally check a field exists."""
    assert result is not None, 'Tool returned no result'
    if isinstance(result, dict):
        if result.get('status') == 'error':
            raise AssertionError(f'Tool error: {result.get("message", result)}')
        if field:
            assert field in result, f'Expected field "{field}" missing from result: {result}'
    print('[OK]')


async def _wait_for_server(url: str, timeout: int = 120, interval: float = 1.0) -> bool:
    """Poll the MCP endpoint until it responds or the timeout expires."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            async with _httpx.AsyncClient() as http:
                await http.get(url, timeout=2.0,
                               headers={'Accept': 'application/json, text/event-stream'})
                return True
        except (_httpx.ConnectError, _httpx.ConnectTimeout, _httpx.ReadTimeout):
            await asyncio.sleep(interval)
    return False


print('Helpers loaded')


## 5  Server startup

Starts the Radiant RAG MCP server in a background daemon thread.
Tries `http` then `streamable-http` transport automatically.
Polls the endpoint for up to 90 s before raising `TimeoutError`.


In [ ]:
import subprocess, time as _time

# Kill any process already bound to the port so restarts work cleanly
_port = int(os.environ['RADIANT_PORT'])
subprocess.run(['fuser', '-k', f'{_port}/tcp'], capture_output=True)
_time.sleep(1.0)

logging.basicConfig(stream=__import__('sys').stderr, level=logging.WARNING, force=True)

_server_ready   = threading.Event()
_server_error   = None
_transport_used = None


def _run_server():
    global _server_error, _transport_used
    try:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        from radiant_rag_mcp.server import mcp
        print('Package  : radiant_rag_mcp  ok')
        _server_ready.set()
        host = os.environ['RADIANT_HOST']
        port = int(os.environ['RADIANT_PORT'])
        for _t in ['http', 'streamable-http']:
            try:
                _transport_used = _t
                mcp.run(transport=_t, host=host, port=port)
                return
            except Exception as _e:
                if 'Unknown transport' in str(_e) or 'unknown transport' in str(_e).lower():
                    continue
                raise
        raise RuntimeError('No supported HTTP transport found (tried: http, streamable-http)')
    except Exception as exc:
        _server_error = exc
        if not _server_ready.is_set():
            _server_ready.set()


_thread = threading.Thread(target=_run_server, name='radiant-mcp', daemon=True)
_thread.start()

if not _server_ready.wait(timeout=30):
    raise TimeoutError('Server thread did not signal ready within 30 s.')
if _server_error:
    raise _server_error

print(f'Waiting for server at {SERVER_URL} ...')
_deadline = time.time() + 90
while time.time() < _deadline:
    if _server_error:
        raise RuntimeError(f'Server thread failed: {_server_error}')
    if asyncio.run(_wait_for_server(SERVER_URL, timeout=5)):
        print(f'Server ready  ->  {SERVER_URL}  (transport: {_transport_used})')
        break
    time.sleep(1)
else:
    raise TimeoutError('Server did not bind within 90 s.')


## 6  Tool list verification

Lists every registered MCP tool and asserts that `ingest_audio` is present.
A missing tool means the audio sub-package did not install or register correctly.


In [ ]:
async def _list_tools():
    async with Client(SERVER_URL) as client:
        return await client.list_tools()

tools      = asyncio.run(_list_tools())
registered = {t.name for t in tools}

print(f'{len(tools)} tool(s) registered:')
for t in sorted(tools, key=lambda x: x.name):
    desc = (t.description or '').splitlines()[0]
    print(f'  {t.name:<30}  {desc}')

print()
if 'ingest_audio' in registered:
    print('[ASSERT OK] ingest_audio is registered')
else:
    print('[FAIL] ingest_audio is NOT in the tool list.')
    print()
    print('  ► Re-run section 1, restart the runtime, and re-run sections 1-6.')
    raise AssertionError('ingest_audio not registered. See diagnostic above.')


## 7  Create synthetic test audio files

Generates three audio files entirely in-process using `espeak` TTS and `ffmpeg`:

| File | Format | Content |
|------|--------|--------|
| `lecture_en.wav` | WAV 16 kHz mono | Machine-learning lecture narration (~30 s) |
| `lecture_en.mp3` | MP3 128 kbps | Same content re-encoded via ffmpeg |
| `lecture_en.m4a` | AAC / M4A | Same content re-encoded via ffmpeg |

All three files share the same narration so that search results in section 11
appear across multiple sources.


In [ ]:
import subprocess
from pathlib import Path

AUDIO_DIR   = Path('/tmp/radiant_audio_test')
AUDIO_DIR.mkdir(exist_ok=True)

WAV_FILE  = str(AUDIO_DIR / 'lecture_en.wav')
MP3_FILE  = str(AUDIO_DIR / 'lecture_en.mp3')
M4A_FILE  = str(AUDIO_DIR / 'lecture_en.m4a')

# Narration text — long enough to produce multiple Whisper segments and chunks
_NARRATION = (
    'Introduction to machine learning. '
    'Machine learning is a branch of artificial intelligence that enables computers '
    'to learn from data without being explicitly programmed. '
    'There are three main paradigms: supervised learning, unsupervised learning, '
    'and reinforcement learning. '
    'In supervised learning, models are trained on labelled examples. '
    'The model learns to map inputs to outputs by minimising a loss function. '
    'Common supervised tasks include classification and regression. '
    'Unsupervised learning discovers hidden patterns in unlabelled data. '
    'Clustering algorithms group similar data points together. '
    'Dimensionality reduction techniques such as principal component analysis '
    'compress data into fewer dimensions while preserving structure. '
    'Reinforcement learning trains an agent to maximise cumulative reward '
    'through interaction with an environment. '
    'Neural networks are the core architecture behind modern machine learning. '
    'A neural network consists of layers of interconnected neurons. '
    'Each neuron applies a weighted sum followed by a non-linear activation function. '
    'Deep learning refers to neural networks with many hidden layers. '
    'Convolutional networks excel at image recognition tasks. '
    'Recurrent networks process sequential data such as time series and text. '
    'Transformers use self-attention mechanisms and power large language models. '
    'Training a neural network requires computing gradients via backpropagation. '
    'Optimisers such as stochastic gradient descent update the weights iteratively. '
    'Regularisation techniques like dropout prevent overfitting to the training set. '
    'Evaluation metrics differ by task: accuracy for classification, '
    'mean squared error for regression, and BLEU score for translation. '
    'This concludes the introduction to machine learning fundamentals.'
)

# ── Generate WAV via espeak TTS ──────────────────────────────────────────
print('Generating WAV with espeak TTS ...')
r = subprocess.run(
    ['espeak', '-s', '140', '-v', 'en', _NARRATION, '-w', WAV_FILE],
    capture_output=True
)
if r.returncode != 0 or not Path(WAV_FILE).exists():
    # fallback: voiced-frequency tones Whisper can detect
    print('  espeak failed — generating voiced-frequency WAV via ffmpeg ...')
    subprocess.run(
        ['ffmpeg', '-y', '-f', 'lavfi',
         '-i', 'aevalsrc=sin(2*PI*200*t)+0.5*sin(2*PI*400*t)+0.3*sin(2*PI*800*t):s=16000:d=30',
         WAV_FILE],
        capture_output=True, check=True
    )
    print('  fallback WAV  OK')
else:
    print('  espeak TTS  OK')

# ── Convert to MP3 ────────────────────────────────────────────────────────
print('Converting to MP3 ...')
subprocess.run(
    ['ffmpeg', '-y', '-i', WAV_FILE, '-codec:a', 'libmp3lame', '-b:a', '128k', MP3_FILE],
    capture_output=True, check=True
)
print('  MP3  OK')

# ── Convert to M4A / AAC ─────────────────────────────────────────────────
print('Converting to M4A ...')
subprocess.run(
    ['ffmpeg', '-y', '-i', WAV_FILE, '-codec:a', 'aac', '-b:a', '128k', M4A_FILE],
    capture_output=True, check=True
)
print('  M4A  OK')

print()
print('Test audio files created:')
for fp in [WAV_FILE, MP3_FILE, M4A_FILE]:
    p = Path(fp)
    print(f'  {p.name:<25}  {p.stat().st_size // 1024:>5} KB  ->  {fp}')

assert Path(WAV_FILE).exists(), f'WAV not created: {WAV_FILE}'
assert Path(MP3_FILE).exists(), f'MP3 not created: {MP3_FILE}'
assert Path(M4A_FILE).exists(), f'M4A not created: {M4A_FILE}'
print('[ASSERT OK] All three test audio files exist on disk')


## 8  `_extract_audio_metadata()` smoke test

Calls `VideoProcessor._extract_audio_metadata()` and `VideoProcessor.is_audio_file()`
directly (no MCP round-trip) and asserts:
- All three files return `is_audio_file() == True`
- Metadata has `is_silent=False` and `duration > 0`
- An MP4 path returns `is_audio_file() == False`


In [ ]:
from radiant_rag_mcp.ingestion.video_processor import VideoProcessor, AUDIO_EXTENSIONS
from radiant_rag_mcp.config import VideoProcessorConfig

cfg = VideoProcessorConfig()
vp  = VideoProcessor(config=cfg)

print('=== is_audio_file() ===')
audio_cases = [
    (WAV_FILE, True),
    (MP3_FILE, True),
    (M4A_FILE, True),
    ('/tmp/some_video.mp4', False),
]
for path, expected in audio_cases:
    result = vp.is_audio_file(path)
    status = 'ok' if result == expected else 'MISMATCH'
    print(f'  {status}  {Path(path).name:<25}  is_audio_file={result}  (expected {expected})')
    assert result == expected, f'is_audio_file mismatch for {path}: got {result}'

print()
print(f'AUDIO_EXTENSIONS = {sorted(AUDIO_EXTENSIONS)}')

print()
print('=== _extract_audio_metadata() ===')
for fpath in [WAV_FILE, MP3_FILE, M4A_FILE]:
    meta = vp._extract_audio_metadata(fpath)
    p    = Path(fpath)
    ok   = meta.duration > 0 and not meta.is_silent
    print(f'  {"ok" if ok else "FAIL"}  {p.name:<25}  '
          f'duration={meta.duration:.1f}s  is_silent={meta.is_silent}  '
          f'title={meta.title}')
    assert meta.duration > 0,   f'{p.name}: duration=0 — ffprobe may be unavailable'
    assert not meta.is_silent,  f'{p.name}: is_silent=True — audio file should not be silent'

print()
print('[ASSERT OK] is_audio_file() and _extract_audio_metadata() correct for all files')


## 9  `ingest_audio` — WAV and MP3 ingestion  *(Whisper transcription)*

Ingests `lecture_en.wav` and `lecture_en.mp3` via the `ingest_audio` MCP tool.
Both files are routed through `process_local_audio()` → Whisper transcription
→ transcript chunks.  Asserts:
- `sources_processed == 2`
- `audio_sources == 2`
- `chunks_created > 0`


In [ ]:
%%time
print('=== ingest_audio (lecture_en.wav + lecture_en.mp3 — Whisper path) ===')
r = run('ingest_audio', {
    'sources':      [WAV_FILE, MP3_FILE],
    'hierarchical': True,
    'summarize':    False,
})

assert_ok(r, 'sources_processed')

assert r.get('sources_processed') == 2, \
    f'Expected sources_processed=2, got {r.get("sources_processed")}'
assert r.get('audio_sources') == 2, \
    f'Expected audio_sources=2, got {r.get("audio_sources")}'
assert r.get('chunks_created', 0) > 0, \
    f'Expected chunks_created>0, got {r.get("chunks_created")}'

print(f'\nChunks created   : {r.get("chunks_created")}')
print(f'Documents stored : {r.get("documents_stored")}')
print(f'Audio sources    : {r.get("audio_sources")}')
print(f'Errors           : {r.get("errors")}')
print('[ASSERT OK] sources_processed==2, audio_sources==2, chunks_created>0')


In [ ]:
%%time
# Also ingest the M4A to exercise a third audio format
print('=== ingest_audio (lecture_en.m4a — M4A format) ===')
r = run('ingest_audio', {
    'sources':      [M4A_FILE],
    'hierarchical': True,
    'summarize':    False,
})

assert_ok(r, 'sources_processed')
assert r.get('sources_processed') == 1, \
    f'Expected sources_processed=1, got {r.get("sources_processed")}'
assert r.get('audio_sources') == 1, \
    f'Expected audio_sources=1, got {r.get("audio_sources")}'

print(f'\nChunks created   : {r.get("chunks_created")}')
print(f'Audio sources    : {r.get("audio_sources")}')
print(f'Errors           : {r.get("errors")}')
print('[ASSERT OK] M4A ingested successfully')


In [ ]:
# Verify rejection of a non-audio path
print('=== ingest_audio (invalid extension — should be rejected) ===')
r = run('ingest_audio', {
    'sources': ['/tmp/some_video.mp4'],
})

assert isinstance(r, dict), 'Expected dict response'
assert r.get('sources_failed') == 1, \
    f'Expected sources_failed=1 for unsupported extension, got: {r}'
assert any('not a recognised audio format' in e for e in r.get('errors', [])), \
    f'Expected format-rejection error message, got: {r.get("errors")}'

print('[ASSERT OK] Non-audio path correctly rejected before processing')


## 10  Index stats after ingestion

Confirms the vector store and BM25 index grew after all three audio files
were ingested in section 9.


In [ ]:
print('=== get_index_stats (after audio ingest) ===')
r = run('get_index_stats')
assert_ok(r)

if isinstance(r, dict):
    health = r.get('health', r)
    vi = health.get('vector_index', {})
    bi = health.get('bm25_index', {})
    print(f'\nVector doc count : {vi.get("document_count", "?")}')
    print(f'BM25   doc count : {bi.get("document_count", "?")}')
    assert (vi.get('document_count') or 0) > 0, \
        'Vector index is empty after ingestion'
    print('[ASSERT OK] Index contains documents')


## 11  `search_knowledge` across audio content

Runs hybrid and BM25 searches against the ingested transcript chunks.
Results should reference content from the machine-learning lecture narration.


In [ ]:
%%time
print('=== search_knowledge: hybrid ===')
r = run('search_knowledge', {
    'query': 'neural network layers neurons activation function',
    'mode':  'hybrid',
    'top_k': 6,
})
assert_ok(r)

if isinstance(r, dict) and r.get('results'):
    print()
    print(f'  {"content_type":<24}  {"source":<25}  {"t/window":<10}  preview')
    print('  ' + '-' * 80)
    for hit in r['results'][:6]:
        meta    = hit.get('meta', hit.get('metadata', {}))
        ctype   = meta.get('content_type', '?')
        ts      = meta.get('start_time', '?')
        source  = Path(meta.get('source', '?')).name
        preview = hit.get('content', '')[:80].replace('\n', ' ')
        print(f'  {ctype:<24}  {source:<25}  {str(ts):<10}  {preview}...')


In [ ]:
%%time
print('=== search_knowledge: bm25 ===')
r = run('search_knowledge', {
    'query': 'supervised learning labelled training data classification',
    'mode':  'bm25',
    'top_k': 5,
})
assert_ok(r)

if isinstance(r, dict) and r.get('results'):
    print()
    for hit in r['results'][:4]:
        meta  = hit.get('meta', hit.get('metadata', {}))
        ctype = meta.get('content_type', '?')
        score = hit.get('score', 0)
        src   = Path(meta.get('source', '?')).name
        print(f'  [{ctype}]  {src}  score={score:.4f}')
        print(f'    {hit.get("content", "")[:100]}')
        print()


## 12  `query_knowledge` grounded in audio content  `[LLM]`

> **Requires a configured LLM backend and a valid `OLLAMA_API_KEY`.**

Sends a natural-language question to the full RAG pipeline.
The retrieval context is drawn from the audio transcript chunks.


In [ ]:
%%time
if not skip_llm():
    print('=== query_knowledge (audio-grounded) ===')
    r = run('query_knowledge', {
        'question': (
            'What are the three main paradigms of machine learning described '
            'in the lecture, and how does each one work?'
        ),
        'mode': 'hybrid',
    })
    assert_ok(r, 'answer')
    print()
    print('Answer:')
    print('-' * 60)
    print(r['answer'][:900])
    print('-' * 60)
    print(f'Warnings : {r.get("warnings", [])}')


## 13  `VideoSummarizationAgent` — `brief` summary  `[LLM]`

> **Requires a configured LLM backend.**

Runs `VideoSummarizationAgent` directly on the WAV transcript chunks.
`summary_detail='brief'` produces:
- Chapter summary: 1 short paragraph
- Overall summary: 1 short paragraph


In [ ]:
%%time
if not skip_llm():
    from radiant_rag_mcp.app import create_app
    from radiant_rag_mcp.agents.video_summarization import VideoSummarizationAgent
    from radiant_rag_mcp.config import VideoSummarizationConfig

    # Build app and extract the shared LLM client once; reused in sections 14-16
    _app = create_app(CONFIG_PATH)
    _llm = _app._orchestrator._llm

    # Retrieve transcript chunks for the WAV file
    hits   = _app.search('lecture_en', mode='dense', top_k=50, show_results=False)
    chunks = [
        doc for doc, _ in hits
        if Path(doc.meta.get('source', '')).name == 'lecture_en.wav'
    ]
    print(f'Chunks retrieved for lecture_en.wav: {len(chunks)}')

    if not chunks:
        print('[NOTE] No chunks found for lecture_en.wav — run section 9 first.')
    else:
        cfg        = VideoSummarizationConfig(summary_detail='brief')
        agent      = VideoSummarizationAgent(llm=_llm, config=cfg)
        _res_brief = agent.summarize_video(WAV_FILE, chunks)

        print(f'Title      : {_res_brief.title}')
        print(f'is_silent  : {_res_brief.is_silent}')
        print(f'Duration   : {_res_brief.duration_seconds:.0f}s')
        print(f'Chapters   : {len(_res_brief.chapters)}')
        print(f'Key topics : {_res_brief.key_topics}')
        print()
        print('Overall summary (brief):')
        print('-' * 60)
        print(_res_brief.summary)
        print('-' * 60)
        print(f'Word count (overall) : {len(_res_brief.summary.split())}')


## 14  `VideoSummarizationAgent` — `standard` summary  `[LLM]`

> **Requires a configured LLM backend.**

`summary_detail='standard'`:
- Chapter summary: 1–2 structured paragraphs
- Overall summary: 2–3 paragraphs


In [ ]:
%%time
if not skip_llm():
    cfg          = VideoSummarizationConfig(summary_detail='standard')
    agent        = VideoSummarizationAgent(llm=_llm, config=cfg)
    _res_standard = agent.summarize_video(WAV_FILE, chunks)

    print('Overall summary (standard):')
    print('-' * 60)
    print(_res_standard.summary)
    print('-' * 60)
    print(f'Word count (overall) : {len(_res_standard.summary.split())}')
    print()
    print('Chapter breakdowns:')
    for ch in _res_standard.chapters:
        print(f'  Ch {ch.index + 1}: [{ch.start:.0f}s – {ch.end:.0f}s]  {ch.title}')
        for ln in ch.summary.split('\n')[:4]:
            print(f'    {ln[:110]}')
        print()


## 15  `VideoSummarizationAgent` — `detailed` summary  `[LLM]`

> **Requires a configured LLM backend.**

`summary_detail='detailed'`:
- Chapter summary: 2–3 paragraphs (8–15 sentences)
- Overall summary: 3–4 structured paragraphs


In [ ]:
%%time
if not skip_llm():
    cfg          = VideoSummarizationConfig(summary_detail='detailed')
    agent        = VideoSummarizationAgent(llm=_llm, config=cfg)
    _res_detailed = agent.summarize_video(WAV_FILE, chunks)

    print('Overall summary (detailed):')
    print('-' * 60)
    print(_res_detailed.summary)
    print('-' * 60)
    print(f'Word count (overall) : {len(_res_detailed.summary.split())}')
    print()
    print(f'Chapters : {len(_res_detailed.chapters)}')
    for ch in _res_detailed.chapters:
        print(f'  Ch {ch.index + 1}: [{ch.start:.0f}s – {ch.end:.0f}s]  {ch.title}')


## 16  Summary detail levels — word-count comparison  `[LLM]`

> **Requires a configured LLM backend.**

Runs all three detail levels and prints a side-by-side table covering:
- **Chapter** average word count
- **Overall** total word count
- **Chapter count**


In [ ]:
%%time
if not skip_llm():
    detail_results = {}
    for detail, cached in [
        ('brief',    globals().get('_res_brief')),
        ('standard', globals().get('_res_standard')),
        ('detailed', globals().get('_res_detailed')),
    ]:
        if cached is not None:
            detail_results[detail] = cached
            print(f'  {detail:<10}  (using cached result from sections 13-15)')
        else:
            cfg   = VideoSummarizationConfig(summary_detail=detail)
            agent = VideoSummarizationAgent(llm=_llm, config=cfg)
            detail_results[detail] = agent.summarize_video(WAV_FILE, chunks)
            print(f'  {detail:<10}  (freshly generated)')

    print()
    HDR = f'  {"Detail":<10}  {"Chapter avg":>12}  {"Overall total":>14}  {"Chapters":>8}'
    print(HDR)
    print('  ' + '-' * (len(HDR) - 2))

    for detail, res in detail_results.items():
        overall_wc = len(res.summary.split())
        ch_wcs     = [len(ch.summary.split()) for ch in res.chapters]
        ch_avg     = (sum(ch_wcs) // max(1, len(ch_wcs))) if ch_wcs else 0
        print(f'  {detail:<10}  {ch_avg:>12,}  {overall_wc:>14,}  {len(res.chapters):>8}')


## 17  Cleanup

- Clears the ChromaDB vector index
- Deletes the temporary audio files from `/tmp/radiant_audio_test/`

Set `RUN_CLEAR = False` to keep the index for further experimentation.


In [ ]:
RUN_CLEAR = True   # Set to False to keep the index intact

if RUN_CLEAR:
    print('=== clear_index ===')
    r = run('clear_index', {'confirm': True})
    if isinstance(r, dict):
        if r.get('cleared'):
            print('[OK] Index cleared.')
        elif ('_ensure_index' in r.get('message', '')
              or 'clear failed' in r.get('message', '').lower()):
            print('[OK] Collection dropped (known ChromaDB reinit pattern).')
        else:
            raise AssertionError(f'Unexpected clear failure: {r}')

    stats = asyncio.run(_call('get_index_stats'))
    if isinstance(stats, dict):
        vi = stats.get('health', stats).get('vector_index', {})
        print(f'Vector doc count post-clear: {vi.get("document_count", "?")}')
else:
    print('RUN_CLEAR=False — index preserved.')


In [ ]:
import shutil

_test_dir = Path('/tmp/radiant_audio_test')
if _test_dir.exists():
    shutil.rmtree(_test_dir)
    print(f'Removed test audio directory: {_test_dir}')
else:
    print(f'Directory not found (already cleaned?): {_test_dir}')

print('Cleanup complete')


## Summary

Run this cell at any point for a status overview.


In [ ]:
print('=' * 65)
print('  Radiant RAG MCP — Audio Ingestion & Summarization Test')
print('  Run complete')
print('=' * 65)
print()
print('  Synthetic test audio  (espeak TTS, ~30 s narration)')
print('    lecture_en.wav  — WAV 16 kHz mono')
print('    lecture_en.mp3  — MP3 128 kbps')
print('    lecture_en.m4a  — AAC / M4A 128 kbps')
print()
print('  Audio metadata')
print('    ok  is_audio_file()           .wav/.mp3/.m4a  == True')
print('    ok  is_audio_file()           .mp4            == False')
print('    ok  _extract_audio_metadata() duration > 0, is_silent=False')
print()
print('  Audio ingestion  (ingest_audio MCP tool)')
print('    ok  lecture_en.wav + .mp3  -> Whisper -> transcript chunks')
print('    ok  lecture_en.m4a         -> Whisper -> transcript chunks')
print('    ok  .mp4 extension         -> rejected before processing')
print()
print('  Search')
print('    ok  search_knowledge hybrid  (content_type=transcript)')
print('    ok  search_knowledge bm25')
print()
print('  LLM cells  [L = requires OLLAMA_API_KEY]')
print('   [L]  query_knowledge grounded in audio transcripts')
print('   [L]  VideoSummarizationAgent  brief')
print('   [L]  VideoSummarizationAgent  standard')
print('   [L]  VideoSummarizationAgent  detailed')
print('   [L]  Word-count table  (chapter avg / overall total)')
print()
print(f'  LLM key available : {HAS_LLM_KEY}')
